# 20개 CNN 실험 결과 통합 분석

이 노트북은 Google Drive의 **`results.zip`** 안에 들어 있는 20개 실험 결과 ZIP을 한 번에 읽어서,
발표에 사용할 수 있는 비교 자료를 자동으로 생성합니다.

## 예상 실험
- Fashion-MNIST: 4개
  - Keras / PyTorch × Adam / SGD
- Furniture: 16개
  - Keras / PyTorch × Adam / SGD × 4단계
  - Baseline → Dropout → SpatialDropout → FC64

## 자동으로 뽑는 자료
- Final / Best Test Accuracy
- Precision / Recall / F1-score
- Best Validation Accuracy / Loss / Epoch
- Final Train / Validation Accuracy 및 Gap
- Final Train / Validation Loss
- 후반 10 Epoch Validation Accuracy 변동폭 / 표준편차
- 후반 10 Epoch Validation Loss 변동폭 / 표준편차
- Parameter 수
- 학습 시간
- Keras / PyTorch 모델 정의 코드 줄 수 / 학습 코드 줄 수
- 클래스별 Precision / Recall / F1
- 여러 비교 그래프
- 전체 결과 CSV
- 분석 결과 ZIP 자동 다운로드

> **중요:** Google Drive의 `내 드라이브/results.zip` 위치에 파일을 올려두고 실행하면 됩니다.

In [ ]:
# ============================================================
# [0] Google Drive 연결 + 경로 설정
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, zipfile, json, re, math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_ZIP = Path("/content/drive/MyDrive/results.zip")

WORK_DIR = Path("/content/results_analysis_work")
EXTRACT_DIR = WORK_DIR / "results"
INNER_EXTRACT_DIR = WORK_DIR / "inner_results"
OUTPUT_DIR = Path("/content/results_analysis_output")
FIG_DIR = OUTPUT_DIR / "figures"

for p in [WORK_DIR, OUTPUT_DIR]:
    if p.exists():
        shutil.rmtree(p)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
INNER_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

if not RESULTS_ZIP.exists():
    raise FileNotFoundError(
        f"results.zip을 찾을 수 없습니다: {RESULTS_ZIP}\n"
        "Google Drive의 내 드라이브 바로 아래에 results.zip을 올렸는지 확인하세요."
    )

print("Results ZIP:", RESULTS_ZIP)
print("Output DIR :", OUTPUT_DIR)

In [ ]:
# ============================================================
# [1] results.zip 압축 해제 + 내부 실험 ZIP 자동 탐색
# ============================================================
with zipfile.ZipFile(RESULTS_ZIP, "r") as zf:
    zf.extractall(EXTRACT_DIR)

inner_zips = sorted(EXTRACT_DIR.rglob("*.zip"))

print(f"내부 ZIP 파일: {len(inner_zips)}개")
for z in inner_zips:
    print(" -", z.name)

# 예상 실험 20개
EXPECTED_EXPERIMENTS = [
    "FashionMNIST_Keras_Adam",
    "FashionMNIST_Keras_SGD",
    "FashionMNIST_PyTorch_Adam",
    "FashionMNIST_PyTorch_SGD",

    "Furniture_Keras_Adam_Baseline",
    "Furniture_Keras_Adam_Dropout",
    "Furniture_Keras_Adam_FC64",
    "Furniture_Keras_Adam_SpatialDropout",

    "Furniture_Keras_SGD_Baseline",
    "Furniture_Keras_SGD_Dropout",
    "Furniture_Keras_SGD_FC64",
    "Furniture_Keras_SGD_SpatialDropout",

    "Furniture_PyTorch_Adam_Baseline",
    "Furniture_PyTorch_Adam_Dropout",
    "Furniture_PyTorch_Adam_FC64",
    "Furniture_PyTorch_Adam_SpatialDropout",

    "Furniture_PyTorch_SGD_Baseline",
    "Furniture_PyTorch_SGD_Dropout",
    "Furniture_PyTorch_SGD_FC64",
    "Furniture_PyTorch_SGD_SpatialDropout",
]

def clean_experiment_name(name):
    # Windows에서 "(1)" 등이 붙은 경우 정리
    name = re.sub(r"\s*\(\d+\)$", "", name)
    return name

# 내부 ZIP을 각각 독립 폴더에 압축 해제
experiment_dirs = {}

for idx, zip_path in enumerate(inner_zips, start=1):
    exp_name = clean_experiment_name(zip_path.stem)
    target = INNER_EXTRACT_DIR / f"{idx:02d}_{exp_name}"
    target.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target)

    experiment_dirs[exp_name] = target

# 혹시 results.zip 안에 ZIP이 아니라 이미 풀린 결과 폴더가 들어있어도 탐색
for summary_path in EXTRACT_DIR.rglob("summary.csv"):
    parent = summary_path.parent
    try:
        summary = pd.read_csv(summary_path).iloc[0]
        exp_name = clean_experiment_name(str(summary.get("experiment", parent.name)))
        if exp_name not in experiment_dirs:
            experiment_dirs[exp_name] = parent
    except Exception:
        pass

found = sorted(experiment_dirs.keys())
missing = sorted(set(EXPECTED_EXPERIMENTS) - set(found))
unexpected = sorted(set(found) - set(EXPECTED_EXPERIMENTS))

print("\n인식된 실험:", len(found), "개")
print("누락:", missing if missing else "없음")
print("예상 외:", unexpected if unexpected else "없음")

if len(found) < 20:
    print("\n[경고] 20개보다 적습니다. results.zip 내부 파일 이름/구조를 확인하세요.")

## [2] 결과 파일 읽기

각 ZIP에서 다음 파일을 우선 사용합니다.

- `summary.csv`
- `history.csv`
- `final_classification_report.csv`
- `best_classification_report.csv`

`summary.csv`에 없는 값은 가능한 경우 classification report / history에서 보완합니다.

In [ ]:
# ============================================================
# [2] 실험별 결과 읽기
# ============================================================

def find_file(exp_dir, filename):
    matches = list(exp_dir.rglob(filename))
    return matches[0] if matches else None

def safe_float(value):
    try:
        if pd.isna(value):
            return np.nan
        return float(value)
    except Exception:
        return np.nan

def infer_fields(exp_name):
    dataset = "Fashion-MNIST" if exp_name.startswith("FashionMNIST_") else "Furniture"
    framework = "Keras" if "_Keras_" in exp_name else "PyTorch"
    optimizer = "Adam" if "_Adam" in exp_name else "SGD"

    if dataset == "Fashion-MNIST":
        version = "CommonCNN"
    else:
        if exp_name.endswith("_Baseline"):
            version = "Baseline"
        elif exp_name.endswith("_Dropout"):
            version = "Dropout"
        elif exp_name.endswith("_SpatialDropout"):
            version = "SpatialDropout"
        elif exp_name.endswith("_FC64"):
            version = "FC64"
        else:
            version = "Unknown"

    return dataset, framework, optimizer, version

master_rows = []
history_map = {}
final_report_map = {}
best_report_map = {}

for exp_name, exp_dir in sorted(experiment_dirs.items()):
    summary_path = find_file(exp_dir, "summary.csv")
    history_path = find_file(exp_dir, "history.csv")
    final_report_path = find_file(exp_dir, "final_classification_report.csv")
    best_report_path = find_file(exp_dir, "best_classification_report.csv")

    if summary_path is None:
        print("[SKIP] summary.csv 없음:", exp_name)
        continue

    sm = pd.read_csv(summary_path).iloc[0].to_dict()
    dataset, framework, optimizer, version = infer_fields(exp_name)

    # summary에 값이 있으면 우선 사용
    row = dict(sm)
    row["experiment"] = clean_experiment_name(str(sm.get("experiment", exp_name)))
    row["dataset"] = sm.get("dataset", dataset)
    row["framework"] = sm.get("framework", framework)
    row["optimizer"] = sm.get("optimizer", optimizer)
    row["model_version"] = sm.get("model_version", version)

    # History
    hist = None
    if history_path is not None:
        hist = pd.read_csv(history_path)

        # epoch 열이 Unnamed로 저장된 경우 정리
        if "epoch" not in hist.columns:
            unnamed = [c for c in hist.columns if c.startswith("Unnamed")]
            if unnamed:
                hist = hist.rename(columns={unnamed[0]: "epoch"})
            else:
                hist.insert(0, "epoch", np.arange(1, len(hist) + 1))

        history_map[row["experiment"]] = hist

        # 마지막 Epoch loss
        if "loss" in hist.columns:
            row["final_train_loss"] = safe_float(hist.iloc[-1]["loss"])
        if "val_loss" in hist.columns:
            row["final_val_loss"] = safe_float(hist.iloc[-1]["val_loss"])

        # 후반 10 Epoch 안정성
        tail = hist.tail(min(10, len(hist)))

        if "val_accuracy" in tail.columns:
            vals = tail["val_accuracy"].astype(float)
            row["last10_val_acc_range_pp"] = (vals.max() - vals.min()) * 100
            row["last10_val_acc_std_pp"] = vals.std(ddof=0) * 100

        if "val_loss" in tail.columns:
            vals = tail["val_loss"].astype(float)
            row["last10_val_loss_range"] = vals.max() - vals.min()
            row["last10_val_loss_std"] = vals.std(ddof=0)

    # Classification report 보완
    if final_report_path is not None:
        final_report = pd.read_csv(final_report_path, index_col=0)
        final_report_map[row["experiment"]] = final_report

        if "macro avg" in final_report.index:
            m = final_report.loc["macro avg"]
            row.setdefault("final_macro_precision", safe_float(m.get("precision")))
            row.setdefault("final_macro_recall", safe_float(m.get("recall")))
            row.setdefault("final_macro_f1", safe_float(m.get("f1-score")))

    if best_report_path is not None:
        best_report = pd.read_csv(best_report_path, index_col=0)
        best_report_map[row["experiment"]] = best_report

        if "macro avg" in best_report.index:
            m = best_report.loc["macro avg"]
            row.setdefault("best_macro_precision", safe_float(m.get("precision")))
            row.setdefault("best_macro_recall", safe_float(m.get("recall")))
            row.setdefault("best_macro_f1", safe_float(m.get("f1-score")))

    # Best - Final 차이
    if "best_test_accuracy" in row and "final_test_accuracy" in row:
        row["best_minus_final_test_acc_pp"] = (
            safe_float(row["best_test_accuracy"]) - safe_float(row["final_test_accuracy"])
        ) * 100

    master_rows.append(row)

master_df = pd.DataFrame(master_rows)

# 발표 비교용 정렬 순서
stage_order = {
    "CommonCNN": 0,
    "Baseline": 0,
    "Dropout": 1,
    "SpatialDropout": 2,
    "FC64": 3,
}
master_df["stage_order"] = master_df["model_version"].map(stage_order).fillna(99)

master_df = master_df.sort_values(
    ["dataset", "framework", "optimizer", "stage_order", "experiment"]
).reset_index(drop=True)

print("Master rows:", len(master_df))
display(master_df.head())

In [ ]:
# ============================================================
# [3] 클래스별 Precision / Recall / F1 통합
# ============================================================
class_rows = []

def append_class_rows(exp_name, report, checkpoint):
    meta = master_df.loc[master_df["experiment"] == exp_name]
    if meta.empty:
        return

    meta = meta.iloc[0]

    # summary/macro/accuracy 행 제외
    excluded = {"accuracy", "macro avg", "weighted avg"}

    for class_name, r in report.iterrows():
        if class_name in excluded:
            continue

        # support가 없는 이상한 행 제외
        if "precision" not in r.index:
            continue

        class_rows.append({
            "experiment": exp_name,
            "dataset": meta["dataset"],
            "framework": meta["framework"],
            "optimizer": meta["optimizer"],
            "model_version": meta["model_version"],
            "checkpoint": checkpoint,
            "class_name": class_name,
            "precision": safe_float(r.get("precision")),
            "recall": safe_float(r.get("recall")),
            "f1_score": safe_float(r.get("f1-score")),
            "support": safe_float(r.get("support")),
        })

for exp_name, report in final_report_map.items():
    append_class_rows(exp_name, report, "Final")

for exp_name, report in best_report_map.items():
    append_class_rows(exp_name, report, "Best")

class_metrics_df = pd.DataFrame(class_rows)

print("Class metric rows:", len(class_metrics_df))
display(class_metrics_df.head(10))

In [ ]:
# ============================================================
# [4] Master CSV 저장
# ============================================================
important_columns = [
    "experiment",
    "dataset",
    "framework",
    "optimizer",
    "model_version",
    "epochs",
    "batch_size",
    "learning_rate",
    "momentum",

    "final_test_accuracy",
    "best_test_accuracy",
    "best_minus_final_test_acc_pp",

    "final_macro_precision",
    "final_macro_recall",
    "final_macro_f1",
    "best_macro_precision",
    "best_macro_recall",
    "best_macro_f1",

    "best_val_accuracy",
    "best_val_loss",
    "best_epoch_by_val_loss",

    "final_train_accuracy",
    "final_val_accuracy",
    "final_train_val_gap",
    "final_train_loss",
    "final_val_loss",

    "last10_val_acc_range_pp",
    "last10_val_acc_std_pp",
    "last10_val_loss_range",
    "last10_val_loss_std",

    "parameter_count",
    "training_seconds",
    "model_definition_code_lines",
    "training_code_lines",

    "dropout",
    "spatial_dropout_1",
    "spatial_dropout_2",
    "fc_units",
]

existing_important = [c for c in important_columns if c in master_df.columns]
master_compact_df = master_df[existing_important].copy()

master_df.to_csv(
    OUTPUT_DIR / "master_summary_all_columns.csv",
    index=False,
    encoding="utf-8-sig"
)

master_compact_df.to_csv(
    OUTPUT_DIR / "master_summary_key_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

class_metrics_df.to_csv(
    OUTPUT_DIR / "class_metrics_all.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:")
print(" - master_summary_all_columns.csv")
print(" - master_summary_key_metrics.csv")
print(" - class_metrics_all.csv")

display(master_compact_df)

# [5] Fashion-MNIST 비교 그래프

Fashion-MNIST 4개는 **Framework × Optimizer 비교**가 목적이므로,
구조 변경 그래프가 아니라 다음 자료를 중심으로 생성합니다.

- Final Test Accuracy
- Precision / Recall / F1
- Best Validation Loss
- 학습 시간
- 코드 길이 비교

In [ ]:
# ============================================================
# [5-1] Fashion-MNIST Final Test Accuracy
# ============================================================
fashion = master_df[master_df["dataset"] == "Fashion-MNIST"].copy()

if not fashion.empty:
    fashion["label"] = fashion["framework"] + "\n" + fashion["optimizer"]

    plt.figure(figsize=(8, 5))
    plt.bar(fashion["label"], fashion["final_test_accuracy"] * 100)
    plt.ylabel("Final Test Accuracy (%)")
    plt.title("Fashion-MNIST - Final Test Accuracy")
    plt.ylim(0, 100)
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fashion_final_test_accuracy.png", dpi=170, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# [5-2] Fashion-MNIST Precision / Recall / F1
# ============================================================
if not fashion.empty:
    labels = fashion["framework"] + " " + fashion["optimizer"]
    x = np.arange(len(fashion))
    width = 0.25

    plt.figure(figsize=(10, 5))
    plt.bar(x - width, fashion["final_macro_precision"] * 100, width, label="Precision")
    plt.bar(x, fashion["final_macro_recall"] * 100, width, label="Recall")
    plt.bar(x + width, fashion["final_macro_f1"] * 100, width, label="F1")
    plt.xticks(x, labels, rotation=20)
    plt.ylabel("Score (%)")
    plt.title("Fashion-MNIST - Macro Precision / Recall / F1")
    plt.ylim(0, 100)
    plt.legend()
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fashion_precision_recall_f1.png", dpi=170, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# [5-3] Fashion-MNIST 학습 시간
# ============================================================
if not fashion.empty and "training_seconds" in fashion.columns:
    plt.figure(figsize=(8, 5))
    plt.bar(fashion["label"], fashion["training_seconds"])
    plt.ylabel("Training Time (seconds)")
    plt.title("Fashion-MNIST - Training Time")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fashion_training_time.png", dpi=170, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# [5-4] Fashion-MNIST 코드 길이 비교
# Keras / PyTorch 구현 방식 차이를 보여주는 보조 자료
# ============================================================
if (
    not fashion.empty
    and "model_definition_code_lines" in fashion.columns
    and "training_code_lines" in fashion.columns
):
    code_compare = (
        fashion.groupby("framework")[["model_definition_code_lines", "training_code_lines"]]
        .mean()
        .reset_index()
    )

    x = np.arange(len(code_compare))
    width = 0.34

    plt.figure(figsize=(8, 5))
    plt.bar(
        x - width/2,
        code_compare["model_definition_code_lines"],
        width,
        label="Model Definition"
    )
    plt.bar(
        x + width/2,
        code_compare["training_code_lines"],
        width,
        label="Training Code"
    )
    plt.xticks(x, code_compare["framework"])
    plt.ylabel("Non-comment Code Lines")
    plt.title("Fashion-MNIST - Keras vs PyTorch Code Length")
    plt.legend()
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fashion_code_length.png", dpi=170, bbox_inches="tight")
    plt.show()

# [6] Furniture 16개 통합 비교

Furniture는 **4단계 구조 변경 과정**이 핵심입니다.

여기서는 다음을 생성합니다.

1. 16개 Final Test Accuracy
2. 16개 Precision / Recall / F1
3. Baseline → Dropout → SpatialDropout → FC64 단계별 추세
4. 후반 10 Epoch Validation Accuracy 변동폭
5. Final vs Best Test Accuracy 차이
6. Parameter 수
7. 학습 시간

In [ ]:
# ============================================================
# [6-1] Furniture 16개 Final Test Accuracy
# ============================================================
furniture = master_df[master_df["dataset"] == "Furniture"].copy()

if not furniture.empty:
    furniture["combo"] = furniture["framework"] + " " + furniture["optimizer"]
    furniture["label"] = furniture["combo"] + " / " + furniture["model_version"]

    plot_df = furniture.sort_values("final_test_accuracy", ascending=True)

    plt.figure(figsize=(11, 8))
    plt.barh(plot_df["label"], plot_df["final_test_accuracy"] * 100)
    plt.xlabel("Final Test Accuracy (%)")
    plt.title("Furniture - 16 Experiments Final Test Accuracy")
    plt.xlim(0, 100)
    plt.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "furniture_16_final_test_accuracy.png", dpi=170, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# [6-2] Furniture 16개 Macro Precision / Recall / F1
# ============================================================
if not furniture.empty:
    plot_df = furniture.copy()
    x = np.arange(len(plot_df))
    width = 0.25

    plt.figure(figsize=(16, 6))
    plt.bar(x - width, plot_df["final_macro_precision"] * 100, width, label="Precision")
    plt.bar(x, plot_df["final_macro_recall"] * 100, width, label="Recall")
    plt.bar(x + width, plot_df["final_macro_f1"] * 100, width, label="F1")
    plt.xticks(x, plot_df["label"], rotation=70, ha="right")
    plt.ylabel("Score (%)")
    plt.title("Furniture - Macro Precision / Recall / F1")
    plt.ylim(0, 100)
    plt.legend()
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "furniture_16_precision_recall_f1.png", dpi=170, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# [6-3] Furniture 단계별 Final Test Accuracy 추세
# ============================================================
stage_names = ["Baseline", "Dropout", "SpatialDropout", "FC64"]
combo_names = [
    "Keras Adam",
    "Keras SGD",
    "PyTorch Adam",
    "PyTorch SGD",
]

if not furniture.empty:
    plt.figure(figsize=(10, 6))

    for combo in combo_names:
        fw, opt = combo.split()
        d = furniture[
            (furniture["framework"] == fw)
            & (furniture["optimizer"] == opt)
        ].copy()

        d["model_version"] = pd.Categorical(
            d["model_version"],
            categories=stage_names,
            ordered=True
        )
        d = d.sort_values("model_version")

        plt.plot(
            d["model_version"].astype(str),
            d["final_test_accuracy"] * 100,
            marker="o",
            label=combo
        )

    plt.ylabel("Final Test Accuracy (%)")
    plt.title("Furniture - Structure Change vs Final Test Accuracy")
    plt.ylim(0, 100)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "furniture_stage_final_test_accuracy.png", dpi=170, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# [6-4] Furniture 단계별 Macro F1 추세
# ============================================================
if not furniture.empty:
    plt.figure(figsize=(10, 6))

    for combo in combo_names:
        fw, opt = combo.split()
        d = furniture[
            (furniture["framework"] == fw)
            & (furniture["optimizer"] == opt)
        ].copy()

        d["model_version"] = pd.Categorical(
            d["model_version"],
            categories=stage_names,
            ordered=True
        )
        d = d.sort_values("model_version")

        plt.plot(
            d["model_version"].astype(str),
            d["final_macro_f1"] * 100,
            marker="o",
            label=combo
        )

    plt.ylabel("Macro F1 (%)")
    plt.title("Furniture - Structure Change vs Macro F1")
    plt.ylim(0, 100)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "furniture_stage_macro_f1.png", dpi=170, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# [6-5] 후반 10 Epoch Validation Accuracy 변동폭
# 낮을수록 후반 학습이 안정적이라고 볼 수 있는 보조 지표
# ============================================================
if not furniture.empty and "last10_val_acc_range_pp" in furniture.columns:
    plt.figure(figsize=(10, 6))

    for combo in combo_names:
        fw, opt = combo.split()
        d = furniture[
            (furniture["framework"] == fw)
            & (furniture["optimizer"] == opt)
        ].copy()

        d["model_version"] = pd.Categorical(
            d["model_version"],
            categories=stage_names,
            ordered=True
        )
        d = d.sort_values("model_version")

        plt.plot(
            d["model_version"].astype(str),
            d["last10_val_acc_range_pp"],
            marker="o",
            label=combo
        )

    plt.ylabel("Validation Accuracy Range (%p)")
    plt.title("Furniture - Last 10 Epoch Validation Accuracy Range")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "furniture_stage_stability_range.png", dpi=170, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# [6-6] Final vs Best Test Accuracy 차이
# 값이 크면 마지막 Epoch 성능과 Best Checkpoint 성능 차이가 큼
# ============================================================
if not furniture.empty and "best_minus_final_test_acc_pp" in furniture.columns:
    plot_df = furniture.sort_values("best_minus_final_test_acc_pp", ascending=True)

    plt.figure(figsize=(11, 8))
    plt.barh(plot_df["label"], plot_df["best_minus_final_test_acc_pp"])
    plt.xlabel("Best - Final Test Accuracy (%p)")
    plt.title("Furniture - Best Checkpoint Advantage")
    plt.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "furniture_best_minus_final_gap.png", dpi=170, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# [6-7] Parameter 수 비교
# ============================================================
if not furniture.empty and "parameter_count" in furniture.columns:
    plt.figure(figsize=(10, 6))

    for combo in combo_names:
        fw, opt = combo.split()
        d = furniture[
            (furniture["framework"] == fw)
            & (furniture["optimizer"] == opt)
        ].copy()

        d["model_version"] = pd.Categorical(
            d["model_version"],
            categories=stage_names,
            ordered=True
        )
        d = d.sort_values("model_version")

        plt.plot(
            d["model_version"].astype(str),
            d["parameter_count"],
            marker="o",
            label=combo
        )

    plt.ylabel("Parameter Count")
    plt.title("Furniture - Parameter Count by Structure")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "furniture_parameter_count.png", dpi=170, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# [6-8] 학습 시간 비교
# ============================================================
if not furniture.empty and "training_seconds" in furniture.columns:
    plot_df = furniture.sort_values("training_seconds", ascending=True)

    plt.figure(figsize=(11, 8))
    plt.barh(plot_df["label"], plot_df["training_seconds"])
    plt.xlabel("Training Time (seconds)")
    plt.title("Furniture - Training Time")
    plt.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "furniture_training_time.png", dpi=170, bbox_inches="tight")
    plt.show()

# [7] FC64 클래스별 Precision / Recall / F1 비교

최종 구조인 FC64에서 각 Framework / Optimizer가
`almirah / chair / fridge / table / tv`를 어떻게 분류했는지 비교합니다.

이 자료는 **Confusion Matrix보다 숫자를 직접 비교하고 싶을 때** 유용합니다.

In [ ]:
# ============================================================
# [7] Furniture FC64 클래스별 Recall 비교
# ============================================================
fc64_class = class_metrics_df[
    (class_metrics_df["dataset"] == "Furniture")
    & (class_metrics_df["model_version"] == "FC64")
    & (class_metrics_df["checkpoint"] == "Final")
].copy()

if not fc64_class.empty:
    fc64_class["combo"] = fc64_class["framework"] + " " + fc64_class["optimizer"]

    pivot_recall = fc64_class.pivot_table(
        index="class_name",
        columns="combo",
        values="recall"
    ) * 100

    ax = pivot_recall.plot(kind="bar", figsize=(11, 6))
    ax.set_ylabel("Recall (%)")
    ax.set_title("Furniture FC64 - Recall by Class")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "furniture_fc64_class_recall.png", dpi=170, bbox_inches="tight")
    plt.show()

    pivot_f1 = fc64_class.pivot_table(
        index="class_name",
        columns="combo",
        values="f1_score"
    ) * 100

    ax = pivot_f1.plot(kind="bar", figsize=(11, 6))
    ax.set_ylabel("F1 (%)")
    ax.set_title("Furniture FC64 - F1 by Class")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "furniture_fc64_class_f1.png", dpi=170, bbox_inches="tight")
    plt.show()

# [8] 자동 핵심 요약

이 셀은 결과를 임의로 좋다/나쁘다 판단하기보다는,
**수치상 가장 큰/작은 항목을 자동으로 찾아주는 참고용 요약**입니다.

최종 발표 결론은 그래프와 실제 실험 목적을 같이 보고 결정하는 것을 권장합니다.

In [ ]:
# ============================================================
# [8] 수치 기반 자동 요약 TXT 생성
# ============================================================
notes = []

notes.append("=== 20개 CNN 실험 통합 결과 자동 요약 ===\n")

# Fashion
if not fashion.empty:
    best_f = fashion.loc[fashion["final_test_accuracy"].idxmax()]
    notes.append(
        f"[Fashion-MNIST] Final Test Accuracy 최고: "
        f"{best_f['framework']} + {best_f['optimizer']} "
        f"({best_f['final_test_accuracy']*100:.2f}%)"
    )

    if "training_seconds" in fashion.columns:
        fastest_f = fashion.loc[fashion["training_seconds"].idxmin()]
        notes.append(
            f"[Fashion-MNIST] 학습 시간 최소: "
            f"{fastest_f['framework']} + {fastest_f['optimizer']} "
            f"({fastest_f['training_seconds']:.1f}초)"
        )

# Furniture
if not furniture.empty:
    best_final = furniture.loc[furniture["final_test_accuracy"].idxmax()]
    notes.append(
        f"\n[Furniture] Final Test Accuracy 최고: "
        f"{best_final['framework']} + {best_final['optimizer']} + {best_final['model_version']} "
        f"({best_final['final_test_accuracy']*100:.2f}%)"
    )

    if "last10_val_acc_range_pp" in furniture.columns:
        stable = furniture.loc[furniture["last10_val_acc_range_pp"].idxmin()]
        unstable = furniture.loc[furniture["last10_val_acc_range_pp"].idxmax()]

        notes.append(
            f"[Furniture] 후반 10 Epoch Validation Accuracy 변동폭 최소: "
            f"{stable['framework']} + {stable['optimizer']} + {stable['model_version']} "
            f"({stable['last10_val_acc_range_pp']:.2f}%p)"
        )

        notes.append(
            f"[Furniture] 후반 10 Epoch Validation Accuracy 변동폭 최대: "
            f"{unstable['framework']} + {unstable['optimizer']} + {unstable['model_version']} "
            f"({unstable['last10_val_acc_range_pp']:.2f}%p)"
        )

    if "best_minus_final_test_acc_pp" in furniture.columns:
        gap = furniture.loc[furniture["best_minus_final_test_acc_pp"].idxmax()]
        notes.append(
            f"[Furniture] Best와 Final Test Accuracy 차이 최대: "
            f"{gap['framework']} + {gap['optimizer']} + {gap['model_version']} "
            f"({gap['best_minus_final_test_acc_pp']:.2f}%p)"
        )

    # 조합별 Baseline -> FC64 parameter 감소
    notes.append("\n[FC64 Parameter 변화]")
    for combo in combo_names:
        fw, opt = combo.split()
        d = furniture[
            (furniture["framework"] == fw)
            & (furniture["optimizer"] == opt)
        ]

        base = d[d["model_version"] == "Baseline"]
        fc64 = d[d["model_version"] == "FC64"]

        if not base.empty and not fc64.empty:
            p0 = float(base.iloc[0]["parameter_count"])
            p1 = float(fc64.iloc[0]["parameter_count"])
            reduction = (1 - p1 / p0) * 100

            notes.append(
                f"- {combo}: {int(p0):,} → {int(p1):,} "
                f"(약 {reduction:.2f}% 감소)"
            )

notes_text = "\n".join(notes)

with open(OUTPUT_DIR / "automatic_numeric_summary.txt", "w", encoding="utf-8") as f:
    f.write(notes_text)

print(notes_text)

# [9] 분석 결과 ZIP 자동 다운로드

생성되는 주요 파일:

```text
results_analysis_output/
├─ master_summary_all_columns.csv
├─ master_summary_key_metrics.csv
├─ class_metrics_all.csv
├─ automatic_numeric_summary.txt
└─ figures/
   ├─ Fashion-MNIST 비교 그래프
   └─ Furniture 16개 비교 그래프
```

In [ ]:
# ============================================================
# [9] 분석 결과 전체 ZIP + 자동 다운로드
# ============================================================
import shutil

ANALYSIS_ZIP = shutil.make_archive(
    "/content/CNN_20개_통합분석_결과",
    "zip",
    root_dir=OUTPUT_DIR
)

print("분석 완료")
print("ZIP:", ANALYSIS_ZIP)

from google.colab import files
files.download(ANALYSIS_ZIP)